# SIH26168 Colab Training — AVNetLite on IO-VNBD
Target 95% accuracy (<5% drift). Run all cells in order. T4 GPU recommended.

In [ ]:
!git clone https://github.com/Himanshu121865/sih26168.git
%cd sih26168
!pip install torch --index-url https://download.pytorch.org/whl/cu121 -q
!pip install pandas scipy onnx onnxruntime onnxscript loguru matplotlib scikit-learn -q
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
!python python/download_iovnbd.py --subset Sync
!ls -lh data/iovnbd/*.zip
!python python/preprocess.py --subset 1h --window 200 --stride 10 --hz 100
!ls -lh data/processed/ && cat python/scaler.json | head -20

In [ ]:
!PYTHONPATH=. python python/train_avnet.py --epochs 50 --batch 128 --lr 1e-3 --device cuda
!ls -lh experiments/checkpoints/

In [ ]:
!PYTHONPATH=. python python/eval_drift.py --model experiments/checkpoints/model_avnet_stage1.p --plot reports/drift_plot.png
from IPython.display import Image; Image("reports/drift_plot.png")

In [ ]:
!PYTHONPATH=. python python/export_tflite.py --model experiments/checkpoints/model_avnet_stage1.p --out model.tflite --onnx model.onnx
!ls -lh model.* scaler.json
!zip -r screening.zip model.tflite scaler.json reports/drift_plot.png python/scaler.json
!ls -lh screening.zip